### This notebook should be run after the isochrone_code.ipynb. 
### This notebook aggregates DA demographic to census tract level and produces static choropleth maps for the web page.
### It also analyses the demographic data by distributing dissemination area (DA) counts onto hex grid by area-weighted overlay to calculate population-weighted average travel times by demographic group.
### Finally, this notebook produces a range of statistical data for the research part of the project (e.g., Population-weighted coverage by mode, Equity gap/Quintile gap).

In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import shapely.affinity

## 1. Aggregate DA Demographics to Census Tracts

Loads DA-level census data, calculates demographic counts, aggregates to census tract level via spatial join, and adds population density. Produces `census_tracts_demographics.geojson` for the choropleth maps.

In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
from lib.census import load_da_census, COUNT_COLS

print("Loading DA Census data...")
DA_Census = load_da_census("data/census/DA/Toronto_DAs_.gpkg") #insert your file path
print(f"Loaded {len(DA_Census)} DAs")

print("Loading CT boundaries...")
toronto_bound = gpd.read_file("data/toronto-boundary-wgs84(1)/citygcs_regional_mun_wgs84.shp").to_crs("EPSG:4326")  #insert your file path
ct = gpd.read_file("data/census/CT/CT_bound/lct_000a21a_e.shp")[['CTUID', 'geometry']].to_crs("EPSG:4326") #insert your file path
ct = gpd.clip(ct, toronto_bound)
print(f"Loaded {len(ct)} CT polygons within Toronto")

print("Spatially joining DAs to CTs...")
da_centroids = DA_Census.to_crs("EPSG:4326").copy()
da_centroids['geometry'] = DA_Census.to_crs("EPSG:26917").geometry.centroid.to_crs("EPSG:4326")
da_to_ct = gpd.sjoin(da_centroids[['DAUID', 'geometry']], ct[['CTUID', 'geometry']],
                     how='left', predicate='within')
DA_Census['CTUID'] = da_to_ct['CTUID'].values
print(f"Matched {da_to_ct['CTUID'].notna().sum()} of {len(da_to_ct)} DAs to CTs")

print("Aggregating to CT level...")
ct_agg = DA_Census.groupby('CTUID')[COUNT_COLS].sum().reset_index()
print(f"Aggregated to {len(ct_agg)} census tracts")

# Merging with geometries, dropping unpopulated tracts
ct_final = ct.merge(ct_agg, on='CTUID', how='left')
ct_final[COUNT_COLS] = ct_final[COUNT_COLS].fillna(0)
ct_final = ct_final[ct_final['pop_total'] > 0].copy()

# Calculating percentages
for col in ['visible_minority', 'low_income', 'first_gen_immigrants', 'seniors', 'children']:
    ct_final[f'{col}_pct'] = ((ct_final[col] / ct_final['pop_total']) * 100).clip(upper=100)

# Calculating population density (people per km²)
ct_final['area_km2']    = ct_final.to_crs("EPSG:26917").geometry.area / 1_000_000
ct_final['pop_density'] = ct_final['pop_total'] / ct_final['area_km2']

# Saving (single write, all columns included)
ct_final.to_file("census_tracts_demographics.geojson", driver='GeoJSON')
print(f"\n✓ Saved census_tracts_demographics.geojson — {len(ct_final)} tracts")
print(ct_final[['visible_minority_pct', 'low_income_pct', 'seniors_pct',
                'children_pct', 'first_gen_immigrants_pct', 'pop_density']].describe().round(1))

# Compute quartile breaks (3 breaks -> 4 colour classes) once, reused by the maps below
# pop_density rounded to whole people/km2, percentages to 1 decimal
break_vars = ['visible_minority_pct', 'low_income_pct', 'first_gen_immigrants_pct',
              'seniors_pct', 'children_pct', 'pop_density']

computed_breaks = {}
print("\nQuartile breaks (paste into Svelte configs):")
for col in break_vars:
    decimals = 0 if col == 'pop_density' else 1
    computed_breaks[col] = ct_final[col].quantile([0.25, 0.5, 0.75]).round(decimals).tolist()
    print(f"  {col}: {computed_breaks[col]}")


Loading DA Census data...
Loaded 3743 DAs
Loading CT boundaries...
Loaded 621 CT polygons within Toronto
Spatially joining DAs to CTs...
Matched 3740 of 3743 DAs to CTs
Aggregating to CT level...
Aggregated to 584 census tracts

✓ Saved census_tracts_demographics.geojson — 584 tracts
       visible_minority_pct  low_income_pct  seniors_pct  children_pct  \
count                 584.0           584.0        584.0         584.0   
mean                   52.8            33.2         17.9          13.8   
std                    24.1             6.7          8.0           4.1   
min                     0.0             0.0          2.2           0.0   
25%                    31.4            28.0         14.1          11.7   
50%                    52.1            34.1         17.0          14.1   
75%                    74.4            37.8         20.4          16.3   
max                    98.9            51.4         92.3          31.1   

       first_gen_immigrants_pct  pop_density  
c

## 2. Export Static Choropleth Maps as PNG

Generates one rotated PNG per demographic variable (bearing −17). Copy outputs to `static/maps/` in the repo.

In [5]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import numpy as np
from matplotlib.patches import Patch

ct = gpd.read_file('census_tracts_demographics.geojson')

# Rotating to match bearing -17 (rotate geometry by +17 degrees)
    # Projecting to UTM, rotate, then plot
ct_proj = ct.to_crs('EPSG:26917')
    # Getting centroid for rotation
cx = ct_proj.geometry.centroid.x.mean()
cy = ct_proj.geometry.centroid.y.mean()

import shapely.affinity
    # Function for rotation
def rotate_gdf(gdf, angle_deg, cx, cy):
    gdf = gdf.copy()
    gdf['geometry'] = gdf.geometry.apply(
        lambda g: shapely.affinity.rotate(g, angle_deg, origin=(cx, cy))
    )
    return gdf

ct_rotated = rotate_gdf(ct_proj, -17, cx, cy)

# Coding the variables for the choropleth maps. Colours are chosen based on the UofT branding scheme.
variables = [
    {
    'col': 'pop_density',
    'title': 'Population Density (people/km²)',
    'colors': ['#fee5d9','#fcbba1','#fc9272','#de2d26'],
    },
    {
        'col': 'visible_minority_pct',
        'title': 'Visible Minority',
        'colors': ['#F0E9F1','#C4A7C9','#9865A1','#6D247A'],
    },
    {
        'col': 'low_income_pct',
        'title': 'Low Income',
        'colors': ['#FBECEA','#F1B5AD','#E67D70','#DC4633'],
    },
    {
        'col': 'first_gen_immigrants_pct',
        'title': 'First Generation Immigrants',
        'colors': ['#E5F5F3','#99D9CF','#4CBDAC','#00A189'],
    },
    {
        'col': 'seniors_pct',
        'title': 'Seniors (65+)',
        'colors': ['#E8EBEF','#A5AFC1','#617393','#1E3765'],
    },
    {
        'col': 'children_pct',
        'title': 'Children (0–14)',
        'colors': ['#F3F8EA','#D1E5AB','#AFD26C','#8DBF2E'],
    },
]

# Attaching computed breaks (from the aggregation cell) to each variable
for v in variables:
    v['breaks'] = computed_breaks[v['col']]

for v in variables:
    colors = v['colors']
    breaks = [0] + v['breaks'] + [100]

    def get_color(val):
        for i in range(len(breaks) - 1):
            if val <= breaks[i + 1]:
                return colors[min(i, len(colors) - 1)]
        return colors[-1]

    ct_rotated['_color'] = ct[v['col']].apply(get_color)

    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    ct_rotated.plot(
        ax=ax,
        color=ct_rotated['_color'],
        edgecolor='white',
        linewidth=0.3
    )

    ax.axis('off')
    ax.set_aspect('equal')

    # Removing all margins
    ax.margins(0)
    plt.tight_layout(pad=0)

    # Legend — bottom left
    labels = [f'< {v["breaks"][0]}%'] + \
             [f'{v["breaks"][i]}–{v["breaks"][i+1]}%'
              for i in range(len(v["breaks"]) - 1)] + \
             [f'> {v["breaks"][-1]}%']

    patches = [Patch(facecolor=c, edgecolor='white', label=l)
               for c, l in zip(colors, labels)]

    legend = ax.legend(
        handles=patches,
        loc='lower right',
        fontsize=9,
        title=v['title'],
        title_fontsize=10,
        framealpha=1.0,
        edgecolor='#cccccc',
        borderpad=0.8,
        handlelength=1.5,
        handleheight=1.5,
    )
    legend.get_title().set_fontweight('bold')

    fname = f'static_map_{v["col"]}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    print(f"Saved {fname}")

Saved static_map_pop_density.png
Saved static_map_visible_minority_pct.png
Saved static_map_low_income_pct.png
Saved static_map_first_gen_immigrants_pct.png
Saved static_map_seniors_pct.png
Saved static_map_children_pct.png

Done. Copy to static/maps/ in your repo.


## 3. Build hex demographic file (DA counts distributed onto hex grid)

Distributes 2021 Census DA counts onto the 200 m hex grid by area-weighted overlay, preserving the isochrone hex order. Produces `hex_with_demographic_counts_fixed.geojson`. Requires `hex_grid.geojson` exported from the isochrone notebook.

In [2]:
# Distribute DA counts onto hexes, preserving gdf_hex (isochrone) order
import geopandas as gpd
import pandas as pd
import numpy as np
from lib.census import load_da_census, COUNT_COLS

hex_grid = gpd.read_file('hex_grid.geojson').to_crs('EPSG:26917') #this file should have been created in the isochrone notebook

DA_Census = load_da_census('data/census/DA/Toronto_DAs_.gpkg').to_crs('EPSG:26917') #insert your file path

DA_Census['da_area'] = DA_Census.geometry.area

overlay = gpd.overlay(
    hex_grid[['id', 'geometry']],
    DA_Census[['DAUID', 'da_area'] + COUNT_COLS + ['geometry']],
    how='intersection'
)
overlay['frac'] = overlay.geometry.area / overlay['da_area']
for col in COUNT_COLS:
    overlay[col] = overlay[col] * overlay['frac']

hex_counts = overlay.groupby('id')[COUNT_COLS].sum().reset_index()

# Merge onto gdf_hex by integer id and SORT BY id to restore isochrone order
hex_final = hex_grid.merge(hex_counts, on='id', how='left').sort_values('id').reset_index(drop=True)
hex_final[COUNT_COLS] = hex_final[COUNT_COLS].fillna(0)

hex_final = hex_final.to_crs('EPSG:4326')
hex_final.to_file('hex_with_demographic_counts_fixed.geojson', driver='GeoJSON')
print(f"Saved {len(hex_final)} hexes in isochrone order")


Saved 6017 hexes in isochrone order


## 4. Population-Weighted Average Travel Time by Demographic Group

Uses transit isochrones (weekday + weekend averaged). Weights each hex's travel time by the demographic population it contains.

In [6]:
# Load hex demographics + window travel times
import geopandas as gpd
import pandas as pd
import numpy as np
from lib.hexdata import load_hex_with_times, DEMOG_GROUPS
from lib.stats import weighted_mean, weighted_percentile

hex_demog = load_hex_with_times()

# Population-weighted mean, median, and IQR per demographic group
rows = []
for label, col in DEMOG_GROUPS.items():
    w = hex_demog[col].values
    for mode in ['transit', 'walk']:
        vals = hex_demog[mode].values
        rows.append({
            'group':  label,
            'mode':   mode,
            'mean':   round(weighted_mean(vals, w), 1),
            'median': round(weighted_percentile(vals, w, 50), 1),
            'p25':    round(weighted_percentile(vals, w, 25), 1),
            'p75':    round(weighted_percentile(vals, w, 75), 1),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv('weighted_stats_by_group.csv', index=False)
print("\nSaved weighted_stats_by_group.csv")

# JS-ready means for DotPlot.svelte
print("\nFor DotPlot.svelte:")
for label in DEMOG_GROUPS:
    t = summary[(summary['group']==label) & (summary['mode']=='transit')]['mean'].iloc[0]
    wa = summary[(summary['group']==label) & (summary['mode']=='walk')]['mean'].iloc[0]
    print(f"  {{ label: '{label}', transit: {t}, walk: {wa} }},")


                group    mode  mean  median  p25  p75
     Total Population transit  16.9    16.5 12.0 22.0
     Total Population    walk  20.9    19.0 12.0 28.0
     Visible Minority transit  17.3    17.0 12.0 22.0
     Visible Minority    walk  21.8    20.0 12.0 29.0
           Low Income transit  16.9    16.5 12.0 22.0
           Low Income    walk  21.1    19.0 12.0 28.0
First Gen. Immigrants transit  17.2    17.0 12.0 22.0
First Gen. Immigrants    walk  21.5    19.0 12.0 29.0
        Seniors (65+) transit  17.3    17.0 12.0 22.5
        Seniors (65+)    walk  21.6    19.0 12.0 29.0
      Children (0–14) transit  17.1    17.0 12.0 22.0
      Children (0–14)    walk  21.3    19.0 12.0 28.0

Saved weighted_stats_by_group.csv

For DotPlot.svelte:
  { label: 'Total Population', transit: 16.9, walk: 20.9 },
  { label: 'Visible Minority', transit: 17.3, walk: 21.8 },
  { label: 'Low Income', transit: 16.9, walk: 21.1 },
  { label: 'First Gen. Immigrants', transit: 17.2, walk: 21.5 },
  {

## 5. Calculating Population Statistics

Calculating equity gap and quintile gap among equity-seeking groups in library access, compared to the population average

In [7]:
# Hard numbers for the Findings section of the publication
import numpy as np
import pandas as pd
from lib.hexdata import load_hex_with_times, DEMOG_GROUPS, DEMOG_PCT_VARS
from lib.stats import weighted_mean, weighted_share

hex_demog = load_hex_with_times()
pop = hex_demog['pop_total'].values

print("\n" + "="*70)
print("1. EQUITY — per-group means + gap from city average")
print("="*70)
city_t = weighted_mean(hex_demog['transit'].values, pop)
city_w = weighted_mean(hex_demog['walk'].values, pop)
print(f"{'Group':22s} {'transit':>8s} {'walk':>8s} {'Δtransit':>9s} {'Δwalk':>8s}")
for label, col in DEMOG_GROUPS.items():
    w = hex_demog[col].values
    t = weighted_mean(hex_demog['transit'].values, w)
    wa = weighted_mean(hex_demog['walk'].values, w)
    print(f"{label:22s} {t:8.1f} {wa:8.1f} {t-city_t:+9.2f} {wa-city_w:+8.2f}")

print(f"\nMax transit gap between any group and city avg: "
      f"{max(abs(weighted_mean(hex_demog['transit'].values, hex_demog[c].values)-city_t) for c in DEMOG_GROUPS.values()):.2f} min")
print(f"Max walk gap between any group and city avg:    "
      f"{max(abs(weighted_mean(hex_demog['walk'].values, hex_demog[c].values)-city_w) for c in DEMOG_GROUPS.values()):.2f} min")

print("\n" + "="*70)
print("2. QUINTILE GAP — Q1 vs Q5 for each demographic (walk+transit)")
print("="*70)
print(f"{'Group':22s} {'Q1 transit':>11s} {'Q5 transit':>11s} {'gap':>6s} | {'Q1 walk':>8s} {'Q5 walk':>8s} {'gap':>6s}")
for col, label in DEMOG_PCT_VARS.items():
    hex_demog['_pct'] = (hex_demog[col] / hex_demog['pop_total']).clip(upper=1.0)
    hex_demog['_q'] = pd.qcut(hex_demog['_pct'], 5, labels=[1,2,3,4,5])
    q1 = hex_demog[hex_demog['_q']==1]
    q5 = hex_demog[hex_demog['_q']==5]
    q1t = weighted_mean(q1['transit'].values, q1['pop_total'].values)
    q5t = weighted_mean(q5['transit'].values, q5['pop_total'].values)
    q1w = weighted_mean(q1['walk'].values, q1['pop_total'].values)
    q5w = weighted_mean(q5['walk'].values, q5['pop_total'].values)
    print(f"{label:22s} {q1t:11.1f} {q5t:11.1f} {q5t-q1t:+6.1f} | {q1w:8.1f} {q5w:8.1f} {q5w-q1w:+6.1f}")



1. EQUITY — per-group means + gap from city average
Group                   transit     walk  Δtransit    Δwalk
Total Population           16.9     20.9     +0.00    +0.00
Visible Minority           17.3     21.8     +0.47    +0.92
Low Income                 16.9     21.1     +0.07    +0.19
First Gen. Immigrants      17.2     21.5     +0.33    +0.67
Seniors (65+)              17.3     21.6     +0.41    +0.70
Children (0–14)            17.1     21.3     +0.26    +0.43

Max transit gap between any group and city avg: 0.47 min
Max walk gap between any group and city avg:    0.92 min

2. QUINTILE GAP — Q1 vs Q5 for each demographic (walk+transit)
Group                   Q1 transit  Q5 transit    gap |  Q1 walk  Q5 walk    gap
Visible Minority              16.5        18.5   +2.0 |     19.2     23.7   +4.5
Low Income                    16.2        17.3   +1.1 |     18.6     21.6   +3.0
First Gen. Immigrants         15.2        18.2   +3.0 |     17.4     23.4   +6.1
Seniors (65+)           

## 6. Transit coverage tables (Table 1 and Table 2 for the research paper)

Population-weighted coverage by mode, and weekday-vs-Saturday comparison. Produces `table_coverage_by_mode.csv` and `table_weekday_vs_saturday.csv`.

In [8]:
# Tables for Isochrone findings
import numpy as np
import pandas as pd
from lib.hexdata import load_hex_with_times
from lib.stats import weighted_mean, weighted_percentile, weighted_share

hex_demog = load_hex_with_times()
pop = hex_demog['pop_total'].values

# ── TABLE 1: Coverage + summary by mode ───────────────────────────────────
print("TABLE 1 — Access to nearest library by mode")
print(f"{'Mode':16s} {'<15min':>7s} {'15-30':>7s} {'30+':>7s} {'Mean':>6s} {'Median':>7s}")
rows = []
for label, col in [('Walking', 'walk'),
                   ('Walk + Transit (weekday)', 'weekday'),
                   ('Walk + Transit (Saturday)', 'weekend')]:
    v = hex_demog[col].values
    r = {
        'Mode': label,
        '<15 min':  f"{weighted_share(v,pop,0,15):.0f}%",
        '15-30 min':f"{weighted_share(v,pop,15,30):.0f}%",
        '30+ min':  f"{weighted_share(v,pop,30,999):.0f}%",
        'Mean':     f"{weighted_mean(v,pop):.1f}",
        'Median':   f"{weighted_percentile(v,pop,50):.1f}",
        'IQR':      f"{weighted_percentile(v,pop,75) - weighted_percentile(v,pop,25):.0f}",
    }
    rows.append(r)
    print(f"{label:16s} {r['<15 min']:>7s} {r['15-30 min']:>7s} {r['30+ min']:>7s} "
          f"{r['Mean']:>6s} {r['Median']:>7s}")

pd.DataFrame(rows).to_csv('table_coverage_by_mode.csv', index=False)

# ── TABLE 2: Weekday vs Saturday comparison ───────────────────────────────
print("\n\nTABLE 2 — Weekday vs Saturday transit (near-identical)")
print(f"{'Statistic':20s} {'Weekday':>9s} {'Saturday':>9s} {'Diff':>6s}")
stats_defs = [
    ('% within 15 min', lambda v: weighted_share(v,pop,0,15)),
    ('% within 30 min', lambda v: weighted_share(v,pop,0,30)),
    ('% over 30 min',   lambda v: weighted_share(v,pop,30,999)),
    ('Mean (min)',      lambda v: weighted_mean(v,pop)),
    ('Median (min)',    lambda v: weighted_percentile(v,pop,50)),
    ('25th pct (min)',  lambda v: weighted_percentile(v,pop,25)),
    ('75th pct (min)',  lambda v: weighted_percentile(v,pop,75)),
]
rows2 = []
for name, fn in stats_defs:
    wdv = fn(hex_demog['weekday'].values)
    wev = fn(hex_demog['weekend'].values)
    rows2.append({'Statistic': name, 'Weekday': round(wdv,1),
                  'Saturday': round(wev,1), 'Difference': round(wdv-wev,1)})
    print(f"{name:20s} {wdv:9.1f} {wev:9.1f} {wdv-wev:+6.1f}")

# Per-hex difference summary
diff = np.abs(hex_demog['weekday'].values - hex_demog['weekend'].values)
print(f"\nPer-hex |weekday - Saturday| difference:")
print(f"  Mean: {diff.mean():.2f} min   Median: {np.median(diff):.1f} min   "
      f"Hexes identical: {(diff==0).sum()} of {len(diff)} ({(diff==0).sum()/len(diff)*100:.0f}%)")

pd.DataFrame(rows2).to_csv('table_weekday_vs_saturday.csv', index=False)
print("\n✓ Saved table_coverage_by_mode.csv and table_weekday_vs_saturday.csv")

from IPython.display import display

# Table 1
t1 = pd.DataFrame(rows)
print("Table 1. Access to nearest library by mode")
display(t1)

# Table 2
t2 = pd.DataFrame(rows2)
print("\nTable 2. Weekday vs Saturday transit")
display(t2)

import matplotlib.pyplot as plt

def table_to_png(df, fname, title):
    fig, ax = plt.subplots(figsize=(8, 0.5 + 0.4*len(df)))
    ax.axis('off')
    ax.set_title(title, fontweight='bold', pad=10, fontsize=11)
    tbl = ax.table(cellText=df.values, colLabels=df.columns,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.5)
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")

table_to_png(t1, 'table_coverage.png', 'Access to nearest library by mode')
table_to_png(t2, 'table_weekday_saturday.png', 'Weekday vs Saturday transit')


TABLE 1 — Access to nearest library by mode
Mode              <15min   15-30     30+   Mean  Median
Walking              35%     43%     21%   20.9    19.0
Walk + Transit (weekday)     41%     54%      5%   16.8    16.0
Walk + Transit (Saturday)     40%     54%      6%   16.9    16.0


TABLE 2 — Weekday vs Saturday transit (near-identical)
Statistic              Weekday  Saturday   Diff
% within 15 min           40.7      40.1   +0.6
% within 30 min           94.7      94.5   +0.2
% over 30 min              5.3       5.5   -0.2
Mean (min)                16.8      16.9   -0.1
Median (min)              16.0      16.0   +0.0
25th pct (min)            12.0      12.0   +0.0
75th pct (min)            22.0      22.0   +0.0

Per-hex |weekday - Saturday| difference:
  Mean: 0.88 min   Median: 0.0 min   Hexes identical: 3376 of 5976 (56%)

✓ Saved table_coverage_by_mode.csv and table_weekday_vs_saturday.csv
Table 1. Access to nearest library by mode


,Mode,<15 min,15-30 min,30+ min,Mean,Median,IQR
0,Walking,35%,43%,21%,20.9,19.0,16
1,Walk + Transit (weekday),41%,54%,5%,16.8,16.0,10
2,Walk + Transit (Saturday),40%,54%,6%,16.9,16.0,10



Table 2. Weekday vs Saturday transit


,Statistic,Weekday,Saturday,Difference
0,% within 15 min,40.7,40.1,0.6
1,% within 30 min,94.7,94.5,0.2
2,% over 30 min,5.3,5.5,-0.2
3,Mean (min),16.8,16.9,-0.1
4,Median (min),16.0,16.0,0.0
5,25th pct (min),12.0,12.0,0.0
6,75th pct (min),22.0,22.0,0.0


Saved table_coverage.png
Saved table_weekday_saturday.png


## 7. Dumbbell Chart Data by Demographic Quintile + Exporting Dumbbells as PNG

Assigns hexes to quintiles by demographic share, then computes population-weighted mean and 25th–75th percentile travel times for the lowest (Q1) and highest (Q5) quintiles. Produces `dumbbell_chart_data.csv` that is then used to generate dumbbell charts as PNGs

In [9]:
# Dumbbell Chart Data: Population-weighted travel time by demographic quintile
# Uses averaged weekday + weekend transit isochrones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.hexdata import load_hex_with_times, DEMOG_PCT_VARS
from lib.stats import weighted_mean, weighted_percentile

hex_demog = load_hex_with_times()
hex_demog = hex_demog.rename(columns={'transit': 'transit_minutes', 'walk': 'walk_minutes'})

# Demographic percent-of-population columns
for col in DEMOG_PCT_VARS:
    hex_demog[f'{col}_pct'] = (hex_demog[col] / hex_demog['pop_total']).clip(upper=1.0)

# Citywide average (population-weighted, all hexes) — for the reference line
city_avg = {
    'transit': round(weighted_mean(hex_demog['transit_minutes'].values, hex_demog['pop_total'].values), 1),
    'walk':    round(weighted_mean(hex_demog['walk_minutes'].values,    hex_demog['pop_total'].values), 1),
}

# Calculating stats by demographic quintile
results = []
for col, label in DEMOG_PCT_VARS.items():
    pct_col = f'{col}_pct'
    hex_demog['quintile'] = pd.qcut(hex_demog[pct_col], q=5, labels=[1,2,3,4,5])
    for quintile, q_label in [(1, 'q1'), (5, 'q5')]:
        subset = hex_demog[hex_demog['quintile'] == quintile]
        w = subset['pop_total'].values
        for mode, time_col in [('transit', 'transit_minutes'), ('walk', 'walk_minutes')]:
            vals = subset[time_col].values
            results.append({
                'variable': col,
                'label':    label,
                'quintile': q_label,
                'mode':     mode,
                'mean':     round(weighted_mean(vals, w), 1),
                'p25':      round(weighted_percentile(vals, w, 25), 1),
                'p75':      round(weighted_percentile(vals, w, 75), 1),
                'n_hex':    len(subset),
                'pop':      round(w.sum()),
            })

df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv('dumbbell_chart_data.csv', index=False)
print("\n Saved dumbbell_chart_data.csv")

variables = df['variable'].unique()
var_labels = df.drop_duplicates('variable').set_index('variable')['label'].to_dict()
labels = [var_labels[v] for v in variables]

for mode, title, color, fname in [
    ('transit', 'Walk + Transit', '#6D247A', 'dumbbell_transit_time_window.png'),
    ('walk',    'Walking Only',   '#C4A7C9', 'dumbbell_walk.png'),
]:
    fig, ax = plt.subplots(figsize=(9, 5))

    # Citywide average reference line
    ax.axvline(city_avg[mode], color='#888', linestyle='--', linewidth=1, alpha=0.7, zorder=0)
    ax.text(city_avg[mode], len(variables) - 0.4, f'city avg {city_avg[mode]}',
            fontsize=8, color='#888', ha='center', va='bottom')

    for i, var in enumerate(variables):
        q1 = df[(df['variable']==var) & (df['quintile']=='q1') & (df['mode']==mode)].iloc[0]
        q5 = df[(df['variable']==var) & (df['quintile']=='q5') & (df['mode']==mode)].iloc[0]
        ax.plot([q1['mean'], q5['mean']], [i, i], color='#ddd', linewidth=1.5, zorder=1)
        ax.errorbar(q1['mean'], i,
                    xerr=[[q1['mean']-q1['p25']], [q1['p75']-q1['mean']]],
                    fmt='o', color='#007FA3', ecolor='#007FA3',
                    elinewidth=2, capsize=4, markersize=7, alpha=0.85, zorder=2)
        ax.errorbar(q5['mean'], i,
                    xerr=[[q5['mean']-q5['p25']], [q5['p75']-q5['mean']]],
                    fmt='o', color='#DC4633', ecolor='#DC4633',
                    elinewidth=2, capsize=4, markersize=7, alpha=0.85, zorder=2)

    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel('Minutes')
    ax.set_title(f'{title}\nteal = Q1 (lowest share) · red = Q5 (highest share) · whiskers = 25th–75th percentile',
                 fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(8, 35)
    ax.set_xticks(range(8, 35))
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {fname}")


            variable                 label quintile    mode  mean  p25  p75  n_hex    pop
    visible_minority      Visible Minority       q1 transit  16.5 11.0 20.0   1196 439590
    visible_minority      Visible Minority       q1    walk  19.2 12.0 24.0   1196 439590
    visible_minority      Visible Minority       q5 transit  18.5 13.5 23.0   1099 559281
    visible_minority      Visible Minority       q5    walk  23.7 15.0 31.0   1099 559281
          low_income            Low Income       q1 transit  16.2 11.0 20.0   1196 463840
          low_income            Low Income       q1    walk  18.6 12.0 23.0   1196 463840
          low_income            Low Income       q5 transit  17.3 12.0 22.5   1195 624158
          low_income            Low Income       q5    walk  21.6 12.0 29.0   1195 624158
first_gen_immigrants First Gen. Immigrants       q1 transit  15.2 11.0 18.5   1196 438179
first_gen_immigrants First Gen. Immigrants       q1    walk  17.4 11.0 21.0   1196 438179
first_gen_